# Week 11 Live Coding: Spending the Last \$50 Million

We turn the seven-state map into numbers: a **win probability** per state, **expected electoral votes**, and then an **allocator** that asks where the next dollar buys the most. The one empirical input — how much a dollar moves the margin — we read off a regression.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

sw = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk11_presidential_allocation/data/swing_states.csv')

# Turn an expected margin (points) into a win probability with a simple S-curve.
# The 3.0 is a chosen "spread": it makes a +3 margin -> ~73% and a 0 margin -> 50%,
# matching the slide. It is an ASSUMPTION, not estimated -- a different number reshapes
# the curve and could reshuffle the ranking. (Exactly the kind of knob to disclose.)
def win_prob(margin):
    return 1 / (1 + np.exp(-margin / 3.0))

# The spending effect is read once from the regression (the only empirical input).
# NOTE: ad_spending_effects.csv is SIMULATED for class, calibrated to the small effects
# the literature reports.
ad = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk11_presidential_allocation/data/ad_spending_effects.csv')
coef = smf.ols('margin_shift_pp ~ spend_advantage_m', data=ad).fit().params['spend_advantage_m']

# Expected electoral votes for a spending plan (a dict: state -> $millions).
# NOTE: we maximize EXPECTED EVs as a stand-in for the real goal -- the probability of
# reaching 270 -- which would also weigh the threshold and the correlation across states.
def expected_ev(spending):
    # each state's dollars from the plan (0 if the plan doesn't mention that state):
    extra = np.array([spending.get(s, 0) for s in sw['state']])
    new_margin = sw['margin'].values + coef * extra          # spending moves the margin
    return float((win_prob(new_margin) * sw['electoral_votes'].values).sum())


In [ ]:
sw

## Part 1: Win probability and expected electoral votes

`win_prob` turns a margin into a probability (0 -> 50%, +3 -> \~73%). **Expected EVs** add up each state's electoral votes times your chance of winning it.

In [ ]:
sw['win_prob'] = win_prob(sw['margin'])
print(sw[['state', 'electoral_votes', 'margin', 'win_prob']].round(2).to_string(index=False))
print('\nExpected swing EVs at $0 spending:', round(expected_ev({}), 2))

You're behind in five of seven, so your expected haul (\~37 of 93) is short. Spending is supposed to change that — but by how much?

## Part 2: How much does a dollar buy? (a regression)

The analytics team related each past campaign's **net ad-spending advantage** to its **shift in margin**. That relationship is a regression, and the coefficient is "points of margin per \$1M."  

In [ ]:
reg = smf.ols('margin_shift_pp ~ spend_advantage_m', data=ad).fit()
print(reg.summary().tables[1])

Read the `spend_advantage_m` row like any coefficient: about **+0.07 points per \$1M**, with a 95% CI of roughly **[0.03, 0.11]**. Two honesty points before we build anything on it:

1. **It's small.** \$50M moves a single state's margin only \~3-4 points. You can flip a 1-point race; you cannot buy a 6-point race.
2. **It's uncertain — and maybe smaller than this.** Our classroom data is clean (and **simulated** for this course), but *real* spend-vs-margin data is confounded (campaigns spend where it's already close, so a naive regression **overstates** the effect — the W2/W3/W7 problem). The best experimental evidence on general-election persuasion (Kalla & Broockman 2018) puts the true effect near **zero**. Keep that over the whole exercise.

## Part 3: The allocator — follow the marginal dollar

Where should the *first* \$1 million go? To whichever state it raises expected EVs the most. Let's compare two by hand, then rank all seven.

In [ ]:
# Pennsylvania (19 EVs, nearly tied) vs Nevada (6 EVs, also close):
print('first $1M in PA:', round(expected_ev({'PA': 1}) - expected_ev({}), 3), 'expected EVs')
print('first $1M in NV:', round(expected_ev({'NV': 1}) - expected_ev({}), 3), 'expected EVs')

In [ ]:
# Rank the states by the value of the first $1M, biggest first.
gains = []
for s in sw['state']:
    gains.append(expected_ev({s: 1}) - expected_ev({}))
ranking = pd.DataFrame({'state': sw['state'], 'first_million_gain': gains})
print(ranking.sort_values('first_million_gain', ascending=False).round(3).to_string(index=False))


Pennsylvania tops the list (big *and* close); Nevada is **last** despite being a near-tie, because it's only 6 EVs. Wisconsin is the *closest to zero* of all and still isn't first. The marginal dollar follows **size x closeness**, not closeness alone.

In [ ]:
# Compare three ways to spend the full $50M:
all_in_nevada = {'NV': 50}
even_split    = {s: 50/7 for s in sw['state']}
tipping_point = {'PA': 50}     # everything into the top-ranked state

for name, plan in [('all-in Nevada', all_in_nevada),
                   ('even split', even_split),
                   ('all-in Pennsylvania', tipping_point)]:
    print(f'{name:22s}: expected EVs = {expected_ev(plan):.2f}')

"Spend where it's closest" (Nevada) is the **worst** of the three. Concentrating on the biggest near-tied state wins. With this strictly *linear* model, the optimizer wants to dump everything into one state — a corner solution. That's not how real campaigns behave, and the reason is **diminishing returns**, which you'll add in the problem set. And hanging over all of it: if the true spending coefficient is near zero, none of these plans differ by much at all.